# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hanizakkk/flyrank_working-repo/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question:** Which content items should be prioritized for refresh review,
based only on search-performance signals observable at the time of the
decision?

**Decision this supports:** where a content/SEO team spends limited manual
refresh-review time in a given month, ranked by evidence rather than gut feel.

**Lane:** Refresh / Content Opportunity Scoring.

In [1]:
# See work/notebooks/capstone_full_pipeline.ipynb for the executed run.
print("Question: which content items should be prioritized for refresh review?")

Question: which content items should be prioritized for refresh review?


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** `FlyRank/internship-warehouse` (Hugging Face, gated). **Table:**
`fact_content_daily_performance`, grain = `report_date x client_hash_id x
content_hash_id`. **Date windows used:** March 2026 (dev decision window),
April 2026 (dev outcome window), May 2026 (sealed-test decision window), June
2026 (sealed-test outcome window AND the live-scoring decision window).

**Excluded:** rows where `gsc_data_available` is not TRUE (no reliable
search-console signal for that row); any month outside March-June 2026 (not
needed for this lane); GA4-only enrichment columns beyond `ga4_sessions`,
`sessions_ai`, `scroll_events` (kept the feature set compact rather than
using every available column). Only hashed `client_hash_id`/`content_hash_id`
ever leave the warehouse query - no domains, URLs, or client names.

In [2]:
import json
for f in ["baseline_metrics", "model_metrics", "split_comparison", "sealed_test_result"]:
    with open(f"../outputs/{f}.json") as fh:
        print(f, "->", json.load(fh))

baseline_metrics -> {'base_rate': 0.5914827592731584, 'decision_month': '2026-03', 'n_rows': 158549, 'outcome_month': '2026-04', 'precision_at_100': 0.78, 'precision_at_50': 0.76, 'thresholds': {'ctr_low': 0.0016611295681063123, 'impressions_visible': 246.0, 'position_weak': 10.0}}
model_metrics -> [{'base_rate': 0.6620141722138716, 'model': 'baseline_rule', 'precision_at_50': 0.7}, {'base_rate': 0.6620141722138716, 'model': 'logistic_regression', 'precision_at_50': 0.84}, {'base_rate': 0.6620141722138716, 'model': 'random_forest', 'precision_at_50': 0.86}]
split_comparison -> [{'base_rate': 0.5913276568905708, 'client_overlap': 45, 'precision_at_50': 0.96, 'split': 'random_row_split'}, {'base_rate': 0.6620141722138716, 'client_overlap': 0, 'precision_at_50': 0.86, 'split': 'client_holdout_split'}]
sealed_test_result -> {'base_rate': 0.7282517565732892, 'decision_month': '2026-05', 'n_rows': 184877, 'outcome_month': '2026-06', 'precision_at_50': 0.86}


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions:** a content item that loses impressions from one month to the
next ("future_decline") is a reasonable, if imperfect, proxy for "would have
benefited from review" - it says nothing about *why* impressions dropped.

**Features (8, all decision-month-only aggregates):** `impressions`, `clicks`,
`avg_position`, `ctr`, `ga4_sessions`, `sessions_ai`, `scroll_events`,
`days_active` - see `work/scripts/warehouse_utils.py::FEATURE_COLUMNS`.

**Label:** `future_decline = 1` if outcome-month summed impressions < decision-month
summed impressions, else 0.

**Baseline:** transparent rule - visible (impressions >= median) AND (weak
position >= 10 OR low CTR < median-among-visible) -> score = flag_sum x
impressions.

**Validation design:** client-holdout split (`client_hash_id`, 80/20,
seed=42) for all reported dev/model numbers; a random row split was run only
to measure the leakage this design choice avoids (see ML-09).

**Leakage checks:** see ML-09 - decision/outcome months never share a query,
IDs excluded from features, deliberate leak demonstration confirms the audit
catches an injected leak (precision@50 -> 1.000).

In [3]:
with open("../outputs/leakage_audit.json") as f:
    print(json.load(f))

{'checklist': {'features_only_from_decision_month': True, 'grouped_split_used_for_reported_results': True, 'ids_excluded_from_features': True, 'no_label_or_sibling_columns_in_features': True}, 'leaky_precision_at_50': 1.0}


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

**Same client-holdout test split, same metric (precision@50):**

| Model | Precision@50 |
|---|---|
| Baseline rule | 0.70 |
| Logistic Regression | 0.84 |
| Random Forest (final) | **0.86** |

**Sealed test (May 2026 -> June 2026, run once, untouched during development):**
Precision@50 = **0.86** - matching the dev-split number almost exactly, which
is the strongest evidence in this whole capstone that the model generalizes
forward in time rather than having been overfit to March/April specifically.

In [4]:
with open("../outputs/model_metrics.json") as f:
    print(json.load(f))
with open("../outputs/sealed_test_result.json") as f:
    print(json.load(f))

[{'base_rate': 0.6620141722138716, 'model': 'baseline_rule', 'precision_at_50': 0.7}, {'base_rate': 0.6620141722138716, 'model': 'logistic_regression', 'precision_at_50': 0.84}, {'base_rate': 0.6620141722138716, 'model': 'random_forest', 'precision_at_50': 0.86}]
{'base_rate': 0.7282517565732892, 'decision_month': '2026-05', 'n_rows': 184877, 'outcome_month': '2026-06', 'precision_at_50': 0.86}


## 5. Limitations

*What this work cannot claim.*

- This is an **observed, decision-support** signal, not a causal claim and not
  a prediction of Google's ranking algorithm.
- The label (impressions-based decline) is a proxy; it doesn't know *why* a
  page's impressions moved.
- The 8-feature set is intentionally compact - it omits most GA4 columns and
  any content-characteristic data (word count, topic, freshness) that isn't
  in this warehouse table, so the model only sees search-behavior signals.
- Precision@50 = 0.86 is measured against a 59-66% base rate (most items
  decline at least somewhat month to month in this window), so the honest
  lift over "just guess decline" is real but moderate, not dramatic.
- The 61,029 items below the 20-impression noise floor get no reliable score
  at all - this method has nothing useful to say about low-traffic content.

In [5]:
print("Limitations documented in the markdown cell above.")

Limitations documented in the markdown cell above.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

**June 2026 live queue, 208,636 items scored, no label yet (forward-looking):**

| Action | Count |
|---|---|
| `refresh_priority` | 59,351 |
| `monitor` | 86,035 |
| `protect` | 2,221 |
| `insufficient_data_manual_review` | 61,029 |

Full detail and reason codes: `work/outputs/ranked_recommendations_sample.json`.

In [6]:
import pandas as pd
ranked = pd.read_json("../outputs/ranked_recommendations_sample.json")
print(ranked["action"].value_counts())
ranked.head(10)

action
refresh_priority    2000
Name: count, dtype: int64


,rank,client_hash_id,content_hash_id,model_score,action,reason_code,confidence,impressions,clicks,avg_position,ctr
0,1,client_b10cb2997d0c7c86,content_87eb4619444da89f,0.833023,refresh_priority,weak_position|low_ctr,high,646,0,28.099554,0.000000
1,2,client_810019792c9b8efc,content_826a6d837682df3c,0.833020,refresh_priority,weak_position|low_ctr,high,498,0,10.845758,0.000000
2,3,client_9958f0a7ae1df715,content_572263a160b6c8ab,0.830888,refresh_priority,weak_position|low_ctr,high,510,0,26.276404,0.000000
3,4,client_73cda7b4e4f265ea,content_1f10cce14fa14a4c,0.830485,refresh_priority,low_ctr,high,338,0,9.095340,0.000000
4,5,client_9958f0a7ae1df715,content_d8288f22519f7d53,0.830259,refresh_priority,weak_position|low_ctr,high,633,0,38.063023,0.000000
5,6,client_b10cb2997d0c7c86,content_5d2c0a2568ca0bac,0.828595,refresh_priority,low_ctr,high,1315,0,8.921163,0.000000
6,7,client_73cda7b4e4f265ea,content_ebe1f0da038a3a54,0.826721,refresh_priority,weak_position|low_ctr,high,2283,2,15.898636,0.000876
7,8,client_b10cb2997d0c7c86,content_8b1d28df3ed4e324,0.825458,refresh_priority,weak_position|low_ctr,high,1176,1,38.878251,0.000850
8,9,client_23a62021009f63c4,content_11a15d9c9ff6f9ea,0.824141,refresh_priority,weak_position|low_ctr,high,651,0,30.984887,0.000000
9,10,client_fef1a8f436438636,content_3aba7dc3b43d583f,0.823872,refresh_priority,weak_position|low_ctr,high,926,0,21.630945,0.000000


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Charts: `work/figures/feature_importance.svg`, `work/figures/action_mix.svg`.
Tables: `work/outputs/*.json`, `work/outputs/ranked_recommendations_sample.json`.
Full executed pipeline: `work/notebooks/capstone_full_pipeline.ipynb`.

In [7]:
import os
for f in ["../figures/feature_importance.svg", "../figures/action_mix.svg"]:
    print(f, os.path.exists(f))

../figures/feature_importance.svg True
../figures/action_mix.svg True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
